## Model Pipeline Transform DBT

In [1]:
import os
import pandas as pd
import duckdb
from sqlalchemy import create_engine
import numpy as np
import psycopg2

engine = create_engine(os.getenv("DATABASE_URL"))

conn = engine.raw_connection()
try:
    df = pd.read_sql(
        "SELECT * FROM jjm_customer_loan where type = 'Bag' and status = 1 order by form_id",
        conn
    )
finally:
    conn.close() 

/tmp/ipykernel_4546/2915320555.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


In [2]:
import os
import pandas as pd
import duckdb
from sqlalchemy import create_engine
import numpy as np
import psycopg2
from io import StringIO
from urllib.parse import urlparse

engine = create_engine(os.getenv("DATABASE_URL"))

conn = engine.raw_connection()
try:
    df = pd.read_sql(
        "SELECT * FROM jjm_customer_loan where type = 'Bag' and status = 1 order by form_id",
        conn
    )
finally:
    conn.close() 

/tmp/ipykernel_4546/328813985.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


#### Load Data on Bronze Warehouse 

In [3]:
# =========================
# 1) CONNECT WAREHOUSE
# =========================
url = os.getenv("DATABASE_WAREHOUSE")
p = urlparse(url)

conn = psycopg2.connect(
    host=p.hostname,
    port=p.port,
    user=p.username,
    password=p.password,
    dbname=p.path.lstrip("/")
)
cur = conn.cursor()

# =========================
# 2) CREATE SCHEMA + TABLE
# =========================
cur.execute("CREATE SCHEMA IF NOT EXISTS bronze;")

cur.execute("""
CREATE TABLE IF NOT EXISTS bronze.dbt_jjm_customer_loan (
    form_id               INTEGER,
    transaction_date      TIMESTAMPTZ,
    contract_num          VARCHAR(20),
    data_source           CHAR(20),
    jewelry_type          VARCHAR(100),
    type                  VARCHAR(255),
    brand                 VARCHAR(255),
    model                 VARCHAR(255),
    sub_model             VARCHAR(255),
    size                  VARCHAR(255),
    color                 VARCHAR(255),
    hardware              VARCHAR(255),
    material              VARCHAR(255),
    code                  VARCHAR(255),
    year_stamp_holo_dc    VARCHAR(255),
    product_year          VARCHAR(100),
    condition             TEXT,
    accessory             TEXT,
    accessory_info        TEXT,
    duration              SMALLINT,
    estimate_amount       NUMERIC(20,2),
    received_amount       NUMERIC(20,2),
    picture_url           TEXT,
    status                SMALLINT,
    editor                VARCHAR(100),
    updated_at            TIMESTAMP,
    actual_price          INTEGER,
    actualprice_status    SMALLINT,
    actualprice_editor    VARCHAR(100),
    used_price            SMALLINT,
    condition_backup      TEXT,
    year_letter           TEXT
);
""")

# overwrite bronze (bronze = raw snapshot)
cur.execute("TRUNCATE TABLE bronze.dbt_jjm_customer_loan;")

# =========================
# 3) PREPARE DATAFRAME
# =========================
df = df.copy()

# ---- INTEGER / SMALLINT columns ----
int_like_cols = [
    "form_id",
    "actual_price",
    "estimate_amount",
    "status",
    "actualprice_status",
    "used_price",
    "duration",
]

for col in int_like_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype("Int64")    # pandas nullable int
            .astype("object")   # prevent "1.0"
        )

# =========================
# 4) SERIALIZE TO CSV
# =========================
buffer = StringIO()
df.to_csv(
    buffer,
    index=False,
    header=False,
    na_rep="\\N"   # explicit NULL marker
)
buffer.seek(0)

# =========================
# 5) COPY WITH COLUMN LIST
# =========================
cur.copy_expert(
    """
    COPY bronze.dbt_jjm_customer_loan (
        form_id,
        transaction_date,
        contract_num,
        data_source,
        jewelry_type,
        type,
        brand,
        model,
        sub_model,
        size,
        color,
        hardware,
        material,
        code,
        year_stamp_holo_dc,
        product_year,
        condition,
        accessory,
        accessory_info,
        duration,
        estimate_amount,
        received_amount,
        picture_url,
        status,
        editor,
        updated_at,
        actual_price,
        actualprice_status,
        actualprice_editor,
        used_price,
        condition_backup,
        year_letter
    )
    FROM STDIN
    WITH (FORMAT CSV, NULL '\\N')
    """,
    buffer
)

# =========================
# 6) COMMIT + CLOSE
# =========================
conn.commit()
cur.close()
conn.close()

print("Data successfully loaded into bronze.dbt_jjm_customer_loan.")

Data successfully loaded into bronze.dbt_jjm_customer_loan.


#### แปลงข้อมูล JSON เป็น CSV เพื่อใช้ในการ map color ใน dbt

In [5]:
import json
import pandas as pd

with open("/home/jovyan/data/mapping/synthetic_brand_palette.json") as f:
    data = json.load(f)

rows = []
for brand, colors in data.items():
    for color, segment in colors.items():
        rows.append({
            "brand": brand,
            "color": color,
            "color_segment": segment
        })

df = pd.DataFrame(rows)
df.to_csv("/home/jovyan/data/mapping/csv/synthetic_color_mapping.csv", index=False)


#### แปลงข้อมูล JSON เป็น CSV เพื่อใช้ในการ map brand hermes ใน dbt

In [13]:
import json
import pandas as pd

# โหลดไฟล์ JSON
with open("/home/jovyan/data/mapping/synthetic_year_code_map.json", "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []

for shape_type, mapping in data.items():
    for letter, year in mapping.items():
        rows.append({
            "shape_type": shape_type,
            "letter": letter,
            "year": year
        })

df = pd.DataFrame(rows)

# บันทึกเป็น CSV
df.to_csv("/home/jovyan/data/mapping/csv/synthetic_year_code_mapping.csv", index=False)

#### ตรวจสอบว่า gold (pandas) กับ mart_dbt_jjm_customer_loan เท่ากันไหม 

In [19]:
import pandas as pd
import psycopg2

# ======================================================
# CONNECT
# ======================================================
conn = psycopg2.connect(
    host="192.168.1.111",
    port=5432,
    dbname="jjm_warehouse",
    user="datateam",
    password="jipjipmoneydata"
)

print("Loading tables...")

df_new = pd.read_sql(
    "select * from gold.jjm_customer_loan",
    conn
)

df_old = pd.read_sql(
    "select * from gold.mart_dbt_jjm_customer_loan",
    conn
)

KEY_COL = "form_id"

# ======================================================
# ALIGN ROWS
# ======================================================
common_ids = set(df_new[KEY_COL]) & set(df_old[KEY_COL])

new_common = (
    df_new[df_new[KEY_COL].isin(common_ids)]
    .set_index(KEY_COL)
    .sort_index()
)

old_common = (
    df_old[df_old[KEY_COL].isin(common_ids)]
    .set_index(KEY_COL)
    .sort_index()
)

# align columns
common_cols = sorted(set(new_common.columns) & set(old_common.columns))

new_common = new_common[common_cols]
old_common = old_common[common_cols]

# ======================================================
# OPTIONAL (VERY IMPORTANT)
# normalize กัน false diff
# ======================================================
def normalize(df):
    return (
        df.fillna("")
          .astype(str)
          .apply(lambda c: c.str.strip())
    )

new_common = normalize(new_common)
old_common = normalize(old_common)

# ======================================================
# COMPARE
# ======================================================
diff = new_common.compare(
    old_common,
    keep_shape=False,
    keep_equal=False
)

# ======================================================
# FLATTEN RESULT
# ======================================================
diff_long = (
    diff.stack(0)
    .reset_index()
)

diff_long.columns = [
    "form_id",
    "column",
    "new_value",
    "old_value"
]

print("\n===== DIFFERENCE SAMPLE =====")
print(diff_long.head(20))

print("\nTotal different rows:", diff_long["form_id"].nunique())
print("Total differences:", len(diff_long))

# ======================================================
# SAVE
# ======================================================
diff_long.to_csv("simple_diff_report.csv", index=False)

print("\n✅ DONE")
print("File created: simple_diff_report.csv")

conn.close()


Loading tables...


/tmp/ipykernel_4546/1698936158.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_new = pd.read_sql(
/tmp/ipykernel_4546/1698936158.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_old = pd.read_sql(



===== DIFFERENCE SAMPLE =====
    form_id                column        new_value    old_value
0       294          product_year              NaN          NaN
1       294  product_year_segment              NaN          NaN
2       294           year_letter                Q      Unknown
3       350          product_year              NaN          NaN
4       350  product_year_segment              NaN          NaN
5       350           year_letter                Q      Unknown
6       735          product_year              NaN          NaN
7       735  product_year_segment              NaN          NaN
8       735           year_letter                S      Unknown
9      1627          product_year              NaN          NaN
10     1627  product_year_segment              NaN          NaN
11     1627           year_letter                P      Unknown
12    10288          product_year          Unknown         2019
13    10288  product_year_segment              NaN          NaN
14    102